# Hedge 03 — construction and cost

Construct two Trump-Yes prediction-market hedges using the Phase 1 headline exposure estimate and the November 4 **20:00 UTC (U.S. equity close)** midpoint probability:

- Retail solar exposure: \(Q_{retail}=\$100{,}000\)
- Institutional solar exposure: \(Q_{inst}=\$50{,}000{,}000\)

The requested signed exposure is \(N=Q\Delta E\). Because the headline \(\Delta E\) is negative, \(N<0\); the offsetting hedge therefore **buys \(-N\) Yes shares**. At entry price \(p\), upfront cost is \((-N)p\). If Yes resolves at $1, hedge profit is \((-N)(1-p)\), offsetting the modeled TAN loss \(Q\Delta E(1-p)\).

In [ ]:
from pathlib import Path

import pandas as pd

PHASE1_RESULTS_PATH = Path("data/01_estimate_delta_final_comparison.csv")
POLYMARKET_PATH = Path("data/polymarket_trump_2024_yes_1min.csv")
HEDGE_RESULTS_PATH = Path("data/hedge03_scenarios.csv")
CUTOFF_DATE = pd.Timestamp("2024-11-04")
ENTRY_TIMESTAMP = pd.Timestamp("2024-11-04T20:00:00Z")

SCENARIOS = {
    "Q_retail": 100_000.0,
    "Q_inst": 50_000_000.0,
}

In [ ]:
phase1_results = pd.read_csv(PHASE1_RESULTS_PATH)
polymarket = (
    pd.read_csv(POLYMARKET_PATH, parse_dates=["timestamp_utc"])
    .set_index("timestamp_utc")
    .sort_index()
)

headline = phase1_results.loc[phase1_results["role"].eq("Headline / sizing")]
if len(headline) != 1:
    raise RuntimeError(f"Expected one headline sizing estimate, found {len(headline)}.")

phase1_delta_E = float(headline.iloc[0]["delta_E"])
p_entry_midpoint = float(polymarket["price"].asof(ENTRY_TIMESTAMP))
remaining_probability_move = 1.0 - p_entry_midpoint

assert phase1_delta_E < 0
assert 0 < p_entry_midpoint < 1

{
    "Phase 1 headline ΔE": phase1_delta_E,
    "entry_timestamp_utc": ENTRY_TIMESTAMP.isoformat(),
    "p_entry_midpoint": p_entry_midpoint,
    "remaining move to Yes resolution": remaining_probability_move,
}

In [ ]:
hedges = pd.DataFrame(
    [{"scenario": scenario, "Q": Q} for scenario, Q in SCENARIOS.items()]
)

# Requested signed exposure and the offsetting long-Yes hedge.
hedges["N_equals_Q_times_delta_E"] = hedges["Q"] * phase1_delta_E
hedges["yes_shares_to_buy"] = -hedges["N_equals_Q_times_delta_E"]
hedges["entry_midpoint_probability"] = p_entry_midpoint
hedges["upfront_midpoint_cost"] = hedges["yes_shares_to_buy"] * p_entry_midpoint
hedges["midpoint_cost_as_pct_of_Q"] = hedges["upfront_midpoint_cost"] / hedges["Q"]
hedges["profit_if_yes"] = hedges["yes_shares_to_buy"] * (1 - p_entry_midpoint)
hedges["modeled_TAN_dollar_move_if_yes"] = (
    hedges["Q"] * phase1_delta_E * (1 - p_entry_midpoint)
)
hedges["net_modeled_move_if_yes"] = (
    hedges["profit_if_yes"] + hedges["modeled_TAN_dollar_move_if_yes"]
)

assert hedges["net_modeled_move_if_yes"].abs().max() < 1e-8
hedges.to_csv(HEDGE_RESULTS_PATH, index=False)

print(
    f"Frozen ΔE: {phase1_delta_E:.6f}; "
    f"20:00 UTC midpoint: {p_entry_midpoint:.4f}"
)
print(f"Saved hedge scenarios to {HEDGE_RESULTS_PATH.resolve()}")
hedges

## Hedge sizes and frictionless cost

Using **ΔE = −0.234202** and the **20:00 UTC midpoint p = 0.5785**:

- **Retail ($100,000 exposure)**
  - Requested signed \(N=Q\Delta E\): **−$23,420.20**
  - Offsetting position: buy **23,420.20 Yes shares**
  - Midpoint cost: **$13,548.58** (**13.55% of Q**)
  - Profit at Yes resolution: **$9,871.61**

- **Institutional ($50 million exposure)**
  - Requested signed \(N=Q\Delta E\): **−$11,710,098.80**
  - Offsetting position: buy **11,710,098.80 Yes shares**
  - Midpoint cost: **$6,774,292.15** (**13.55% of Q**)
  - Profit at Yes resolution: **$4,935,806.85**

These are frictionless mark-price costs. They exclude order-book depth, price impact, fees, execution risk, and position limits; those considerations are especially material for the institutional scenario.

## Execution-cost methods

Three diagnostics are applied at the November 4 close:

1. **Quoted half-spread:** unavailable historically. The CLOB price-history endpoint contains only `t` and `p`, and Tsang–Yang (arXiv:2603.03152, §2.3) state that their on-chain data do not observe standing order-book depth or quoted spreads. No paper-reported numerical effective spread was found. A one-tick half-spread (**0.0005**, from the market's 0.001 minimum tick) is shown only as a labeled lower-bound proxy—not as an observed quote.
2. **Effective half-spread:** volume-weighted \(D_i(P_i-M_i)\) from Trump-Yes taker trades within ±30 minutes of the November 4 U.S. close, where `M` is the latest one-minute CLOB price and `D` is +1 for buys and −1 for sells.
3. **Price impact:** \(\Delta\theta=\lambda q\), with \(\lambda=0.01\) log-odds per **$1 million** of flow, consistent with Tsang–Yang (arXiv:2603.03136, §8). Impact is converted back to probability space, and cost assumes linear movement from the entry price to the post-trade price, so average impact is half the endpoint move.

In [ ]:
import math

import requests

CONDITION_ID = "0xdd22472e552920b8438158ea7238bfadfa4f736aa4cee91a6b86c39ead110917"
YES_TOKEN_ID = "21742633143463906290569050155826241533067272736897614950488156847949938836455"
TRADES_URL = "https://data-api.polymarket.com/trades"
PRICE_HISTORY_PATH = Path("data/polymarket_trump_2024_yes_1min.csv")
NEAR_CLOSE_TRADES_PATH = Path("data/hedge03_nov4_close_trades.csv")
EXECUTION_COSTS_PATH = Path("data/hedge03_execution_costs.csv")

WINDOW_START = pd.Timestamp("2024-11-04T19:30:00Z")
WINDOW_END = pd.Timestamp("2024-11-04T20:30:00Z")
MIN_TICK = 0.001
LAMBDA = 0.01  # log-odds per $1 million signed flow

In [ ]:
response = requests.get(
    TRADES_URL,
    params={
        "market": CONDITION_ID,
        "start": int(WINDOW_START.timestamp()),
        "end": int(WINDOW_END.timestamp()),
        "limit": 10_000,
        "takerOnly": "true",
    },
    timeout=60,
)
response.raise_for_status()
trades = pd.DataFrame(response.json())

close_trades = trades.loc[trades["asset"].astype(str).eq(YES_TOKEN_ID)].copy()
close_trades["timestamp_utc"] = pd.to_datetime(close_trades["timestamp"], unit="s", utc=True)

minute_midpoints = (
    pd.read_csv(PRICE_HISTORY_PATH, usecols=["timestamp_utc", "price"], parse_dates=["timestamp_utc"])
    .sort_values("timestamp_utc")
    .rename(columns={"price": "midpoint_proxy"})
)
close_trades = pd.merge_asof(
    close_trades.sort_values("timestamp_utc"),
    minute_midpoints,
    on="timestamp_utc",
    direction="backward",
    tolerance=pd.Timedelta("2min"),
)
close_trades["direction"] = close_trades["side"].map({"BUY": 1, "SELL": -1})
close_trades["effective_half_spread"] = close_trades["direction"] * (
    close_trades["price"] - close_trades["midpoint_proxy"]
)

assert len(close_trades) > 0
assert close_trades["midpoint_proxy"].notna().all()
assert close_trades["direction"].notna().all()

vw_effective_half_spread = (
    close_trades["effective_half_spread"] * close_trades["size"]
).sum() / close_trades["size"].sum()
one_tick_half_spread_lower_bound = MIN_TICK / 2

close_trades[
    ["timestamp_utc", "side", "size", "price", "midpoint_proxy", "effective_half_spread", "transactionHash"]
].to_csv(NEAR_CLOSE_TRADES_PATH, index=False)

print(f"Trump-Yes trades near close: {len(close_trades):,}")
print(f"Share volume: {close_trades['size'].sum():,.2f}")
print(f"VW effective half-spread: {vw_effective_half_spread:.6f}")
print(f"One-tick half-spread lower bound: {one_tick_half_spread_lower_bound:.6f}")

In [ ]:
logit_p = math.log(p_entry_midpoint / (1 - p_entry_midpoint))
execution_costs = hedges[
    ["scenario", "Q", "yes_shares_to_buy", "upfront_midpoint_cost"]
].copy()
execution_costs = execution_costs.rename(
    columns={"upfront_midpoint_cost": "mark_price_cost"}
)
execution_costs["entry_midpoint_probability"] = p_entry_midpoint

# (a) No historical quote is available; retain NA and report a one-tick lower bound separately.
execution_costs["a_quoted_half_spread_cost"] = float("nan")
execution_costs["a_one_tick_lower_bound_cost"] = (
    execution_costs["yes_shares_to_buy"] * one_tick_half_spread_lower_bound
)

# (b) Empirical trade-based effective half-spread near the Nov 4 close.
execution_costs["b_effective_half_spread"] = vw_effective_half_spread
execution_costs["b_effective_spread_cost"] = (
    execution_costs["yes_shares_to_buy"] * vw_effective_half_spread
)

# (c) Kyle impact: order flow is transaction value in $ millions, not raw shares.
execution_costs["order_flow_usd_millions"] = execution_costs["mark_price_cost"] / 1_000_000
execution_costs["impact_delta_log_odds"] = (
    LAMBDA * execution_costs["order_flow_usd_millions"]
)
execution_costs["post_trade_probability"] = execution_costs["impact_delta_log_odds"].map(
    lambda delta: 1 / (1 + math.exp(-(logit_p + delta)))
)
execution_costs["endpoint_probability_impact"] = (
    execution_costs["post_trade_probability"] - p_entry_midpoint
)
execution_costs["c_linear_price_impact_cost"] = (
    execution_costs["yes_shares_to_buy"]
    * execution_costs["endpoint_probability_impact"]
    / 2
)

execution_costs["all_in_cost_effective_spread"] = (
    execution_costs["mark_price_cost"] + execution_costs["b_effective_spread_cost"]
)
execution_costs["all_in_cost_linear_impact"] = (
    execution_costs["mark_price_cost"] + execution_costs["c_linear_price_impact_cost"]
)
execution_costs.to_csv(EXECUTION_COSTS_PATH, index=False)

print(f"Saved execution costs to {EXECUTION_COSTS_PATH.resolve()}")
execution_costs

## Recorded execution costs

The Nov 4 ±30-minute sample contains **3,467 Trump-Yes trades** and **670,324.83 shares**. The volume-weighted effective half-spread is **0.000750** per share (full effective spread ≈ **0.001501**).

- **Retail hedge** — 23,420 Yes shares; 20:00 UTC midpoint cost $13,548.58
  - (a) Historical quoted half-spread: **unavailable**; one-tick lower-bound proxy cost: **$11.71**
  - (b) Effective-spread cost: **$17.57**; all-in: **$13,566.16**
  - (c) Linear λ-impact cost: **$0.39**; all-in: **$13,548.97**

- **Institutional hedge** — 11,710,099 Yes shares; 20:00 UTC midpoint cost $6,774,292.15
  - (a) Historical quoted half-spread: **unavailable**; one-tick lower-bound proxy cost: **$5,855.05**
  - (b) Effective-spread cost: **$8,786.62**; all-in: **$6,783,078.77**
  - (c) Linear λ-impact cost: **$96,167.18**; all-in: **$6,870,459.33**

The institutional order is about 17.5 times the observed one-hour share volume, so extrapolating a constant λ is highly uncertain; actual execution would require slicing and order-book-depth data.

## Same book, opposite verdict

Total friction combines the empirically estimated effective-spread cost with the λ-based linear price-impact cost. The one-tick figure is not added because it is a lower-bound proxy for the same spread component and would double count it.

\[
\text{Total friction as \% of Q}
= \frac{\text{effective-spread cost}+\text{price-impact cost}}{Q}\times 100.
\]

In [ ]:
FRICTION_TABLE_PATH = Path("data/hedge03_total_friction_pct_Q.csv")

execution_costs["total_friction"] = (
    execution_costs["b_effective_spread_cost"]
    + execution_costs["c_linear_price_impact_cost"]
)
execution_costs["total_friction_pct_Q"] = (
    100 * execution_costs["total_friction"] / execution_costs["Q"]
)

friction_by_scenario = execution_costs.set_index("scenario")["total_friction_pct_Q"]
two_cell_table = pd.DataFrame(
    {
        "Q_retail": [friction_by_scenario["Q_retail"]],
        "Q_inst": [friction_by_scenario["Q_inst"]],
    },
    index=["Total friction (% of Q)"],
)
two_cell_table.to_csv(FRICTION_TABLE_PATH, index_label="metric")

print(f"Saved two-cell friction table to {FRICTION_TABLE_PATH.resolve()}")
two_cell_table

## Verdict

- **Q_retail:** total friction = **0.017960% of Q** (**1.80 bps**) — operationally small.
- **Q_inst:** total friction = **0.209908% of Q** (**20.99 bps**) — materially larger and dependent on aggressive impact extrapolation.

The same Polymarket book therefore supports opposite implementation verdicts by scale: plausible for the retail hedge, but execution-constrained for the institutional hedge.

## Final friction deliverable

Historical standing book depth is unavailable, so the scale diagnostic uses actual Trump-Yes share volume traded in the ±30-minute window around the November 4 close. Exceeding this volume is not literally the same as exceeding displayed depth, but it is strong evidence that immediate screen execution would require substantial impact, slicing, or negotiated blocks.

In [ ]:
FINAL_FRICTION_TABLE_PATH = Path("data/hedge03_final_friction_table.csv")
observed_close_window_volume = close_trades["size"].sum()

final_friction_table = execution_costs[
    [
        "scenario",
        "Q",
        "yes_shares_to_buy",
        "b_effective_spread_cost",
        "c_linear_price_impact_cost",
    ]
].copy()
final_friction_table["order_as_multiple_of_1h_traded_volume"] = (
    final_friction_table["yes_shares_to_buy"] / observed_close_window_volume
)
final_friction_table["effective_spread_bps_of_Q"] = (
    10_000 * final_friction_table["b_effective_spread_cost"] / final_friction_table["Q"]
)
final_friction_table["price_impact_bps_of_Q"] = (
    10_000 * final_friction_table["c_linear_price_impact_cost"] / final_friction_table["Q"]
)
final_friction_table["total_friction_bps_of_Q"] = (
    final_friction_table["effective_spread_bps_of_Q"]
    + final_friction_table["price_impact_bps_of_Q"]
)
final_friction_table["impact_share_of_total_friction"] = (
    final_friction_table["c_linear_price_impact_cost"]
    / (
        final_friction_table["b_effective_spread_cost"]
        + final_friction_table["c_linear_price_impact_cost"]
    )
)
final_friction_table["verdict"] = [
    "Single-digit bps; plausible at observed scale",
    "Impact dominates; infeasible without slicing or blocks",
]
final_friction_table.to_csv(FINAL_FRICTION_TABLE_PATH, index=False)

print(f"Observed ±30-minute share volume: {observed_close_window_volume:,.2f}")
print(f"Saved final friction table to {FINAL_FRICTION_TABLE_PATH.resolve()}")
final_friction_table

# Final verdict

- **Retail:** **1.80 bps of Q** total friction. The order is **3.49%** of observed one-hour share volume; spread cost dominates and implementation is plausible.
- **Institutional:** **20.99 bps of Q** total friction. The order is **17.47×** observed one-hour share volume; modeled impact contributes **91.63%** of friction, making immediate execution infeasible without substantial slicing or blocks.

This is the “same book, opposite verdict” result. The institutional scale comparison is against observed trading volume, not unavailable historical displayed depth.